In [20]:
import numpy as np
import meshio

In [13]:
def per_patch_quad_nodes_pinched(p):
    '''
    tri node order:
    0: p300
    1: p030
    2: p003
    3: p210
    4: p120
    5: p021
    6: p012
    7: p102
    8: p201
    9: p111

    convertion:
    third = 1.0/3.0
    Q [0 , 0] = P [0 ,3 ,0]
    Q [0 , 1] = P [1 ,2 ,0]
    Q [0 , 2] = P [2 ,1 ,0]
    Q [0 , 3] = P [3 ,0 ,0]
    Q [1 , 0] = P [0 ,2 ,1]
    Q [1 , 1] = third * ( P [1 ,2 ,0] + 2* P [1 ,1 ,1])
    Q [1 , 2] = third * ( P [2 ,0 ,1] + 2* P [2 ,1 ,0])
    Q [1 , 3] = P [3 ,0 ,0]
    Q [2 , 0] = P [0 ,1 ,2]
    Q [2 , 1] = third * ( P [1 ,0 ,2] + 2* P [1 ,1 ,1])
    Q [2 , 2] = third * ( P [2 ,1 ,0] + 2* P [2 ,0 ,1])
    Q [2 , 3] = P [3 ,0 ,0]
    Q [3 , 0] = P [0 ,0 ,3]
    Q [3 , 1] = P [1 ,0 ,2]
    Q [3 , 2] = P [2 ,0 ,1]
    Q [3 , 3] = P [3 ,0 ,0]

    quad node order:
    0: q00
    1: q30
    2: q33
    3: q03
    4: q10
    5: q20
    6: q31
    7: q32
    8: q23
    9: q13
    10: q02
    11: q01
    12: q11
    13: q21
    14: q22
    15: q12
    '''

    q = [None] * 16
    q[0] = p[1]
    q[1] = p[2]
    q[2] = p[0]
    q[3] = p[0]
    q[4] = p[5]
    q[5] = p[6]
    q[6] = p[7]
    q[7] = p[8]
    q[8] = p[0]
    q[9] = p[0]
    q[10] = p[3]
    q[11] = p[4]
    q[12] = 1./3.*(p[4] + 2.*p[9])
    q[13] = 1./3.*(p[7] + 2.*p[9])
    q[14] = 1./3.*(p[3] + 2.*p[8])
    q[15] = 1./3.*(p[8] + 2.*p[3])

    return q


In [14]:
def per_patch_quad_nodes_trimmed(p):
    q = [None] * 16

    q[0] = p[2]
    q[11] = p[6]
    q[10] = p[5]
    q[3] = p[1]
    q[4] = p[7]
    q[12] = 1./3. *( - p[2]+ p[6]+ p[7]+2* p[9])
    q[15] = 1./3. *( -2* p[6]+2* p[5]+2* p[9]+ p[4])
    q[9] = p[1] - p[5]+ p[4]
    q[5] = p[8]
    q[13] = 1./3. *( -2* p[7]+2* p[9]+2* p[8]+ p[3])
    q[14] = 1./3. *( p[2] -2* p[6]+ p[5] -2* p[7]+2* p[4]+ p[8]+2* p[3])
    q[8] = p[6] -2* p[5]+ p[1] -2* p[9]+2* p[4]+ p[3]
    q[1] = p[0]
    q[6] = p[0] - p[8]+ p[3]
    q[7] = p[7] -2* p[9]+ p[4] -2* p[8]+2* p[3]+ p[0]
    q[2] = -p[2]+3* p[6] -3* p[5]+ p[1]+3* p[7] -6* p[9]+3* p[4] -3* p[8]+3* p[3]+ p[0]

    return q

In [ ]:
def per_patch_quad_nodes(V, f, mode="trimmed"):
    assert f.shape[0] == 10
    p = V[f]

    if mode == "trimmed":
        return per_patch_quad_nodes_trimmed(p)
    elif mode == "pinched":
        return per_patch_quad_nodes_pinched(p)
    else:
        raise Exception("Unsupported convertion mode! Use \"trimmed\" or \"pinched\"")


In [11]:
# trimmed
trimmed_string = "third = 1.0/3.0\n\
    Q [0 ,0] = P [0 ,0 ,3]\n\
    Q [0 ,1] = P [0 ,1 ,2]\n\
    Q [0 ,2] = P [0 ,2 ,1]\n\
    Q [0 ,3] = P [0 ,3 ,0]\n\
    Q [1 ,0] = P [1 ,0 ,2]\n\
    Q [1 ,1] = third *( - P [0 ,0 ,3]+ P [0 ,1 ,2]+ P [1 ,0 ,2]+2* P [1 ,1 ,1])\n\
    Q [1 ,2] = third *( -2* P [0 ,1 ,2]+2* P [0 ,2 ,1]+2* P [1 ,1 ,1]+ P [1 ,2 ,0])\n\
    Q [1 ,3] = P [0 ,3 ,0] - P [0 ,2 ,1]+ P [1 ,2 ,0]\n\
    Q [2 ,0] = P [2 ,0 ,1]\n\
    Q [2 ,1] = third *( -2* P [1 ,0 ,2]+2* P [1 ,1 ,1]+2* P [2 ,0 ,1]+ P [2 ,1 ,0])\n\
    Q [2 ,2] = third *( P [0 ,0 ,3] -2* P [0 ,1 ,2]+ P [0 ,2 ,1] -2* P [1 ,0 ,2]+2* P [1 ,2 ,0]+ P [2 ,0 ,1]+2* P [2 ,1 ,0])\n\
    Q [2 ,3] = P [0 ,1 ,2] -2* P [0 ,2 ,1]+ P [0 ,3 ,0] -2* P [1 ,1 ,1]+2* P [1 ,2 ,0]+ P [2 ,1 ,0]\n\
    Q [3 ,0] = P [3 ,0 ,0]\n\
    Q [3 ,1] = P [3 ,0 ,0] - P [2 ,0 ,1]+ P [2 ,1 ,0]\n\
    Q [3 ,2] = P [1 ,0 ,2] -2* P [1 ,1 ,1]+ P [1 ,2 ,0] -2* P [2 ,0 ,1]+2* P [2 ,1 ,0]+ P [3 ,0 ,0]\n\
    Q [3 ,3] = -P [0 ,0 ,3]+3* P [0 ,1 ,2] -3* P [0 ,2 ,1]+ P [0 ,3 ,0]+3* P [1 ,0 ,2] -6* P [1 ,1 ,1]+3* P [1 ,2 ,0] -3* P [2 ,0 ,1]+3* P [2 ,1 ,0]+ P [3 ,0 ,0]"

trimmed_string = trimmed_string.replace("[3 ,0 ,0]", "[0]")
trimmed_string = trimmed_string.replace("[0 ,3 ,0]", "[1]")
trimmed_string = trimmed_string.replace("[0 ,0 ,3]", "[2]")
trimmed_string = trimmed_string.replace("[2 ,1 ,0]", "[3]")
trimmed_string = trimmed_string.replace("[1 ,2 ,0]", "[4]")
trimmed_string = trimmed_string.replace("[0 ,2 ,1]", "[5]")
trimmed_string = trimmed_string.replace("[0 ,1 ,2]", "[6]")
trimmed_string = trimmed_string.replace("[1 ,0 ,2]", "[7]")
trimmed_string = trimmed_string.replace("[2 ,0 ,1]", "[8]")
trimmed_string = trimmed_string.replace("[1 ,1 ,1]", "[9]")

trimmed_string = trimmed_string.replace("[0 ,0]", "[0]")
trimmed_string = trimmed_string.replace("[3 ,0]", "[1]")
trimmed_string = trimmed_string.replace("[3 ,3]", "[2]")
trimmed_string = trimmed_string.replace("[0 ,3]", "[3]")
trimmed_string = trimmed_string.replace("[1 ,0]", "[4]")
trimmed_string = trimmed_string.replace("[2 ,0]", "[5]")
trimmed_string = trimmed_string.replace("[3 ,1]", "[6]")
trimmed_string = trimmed_string.replace("[3 ,2]", "[7]")
trimmed_string = trimmed_string.replace("[2 ,3]", "[8]")
trimmed_string = trimmed_string.replace("[1 ,3]", "[9]")
trimmed_string = trimmed_string.replace("[0 ,2]", "[10]")
trimmed_string = trimmed_string.replace("[0 ,1]", "[11]")
trimmed_string = trimmed_string.replace("[1 ,1]", "[12]")
trimmed_string = trimmed_string.replace("[2 ,1]", "[13]")
trimmed_string = trimmed_string.replace("[2 ,2]", "[14]")
trimmed_string = trimmed_string.replace("[1 ,2]", "[15]")

trimmed_string = trimmed_string.replace("third", "1./3.")
trimmed_string = trimmed_string.replace("P ", "p")
trimmed_string = trimmed_string.replace("Q ", "q")

In [12]:
print(trimmed_string)

1./3. = 1.0/3.0
    q[0] = p[2]
    q[11] = p[6]
    q[10] = p[5]
    q[3] = p[1]
    q[4] = p[7]
    q[12] = 1./3. *( - p[2]+ p[6]+ p[7]+2* p[9])
    q[15] = 1./3. *( -2* p[6]+2* p[5]+2* p[9]+ p[4])
    q[9] = p[1] - p[5]+ p[4]
    q[5] = p[8]
    q[13] = 1./3. *( -2* p[7]+2* p[9]+2* p[8]+ p[3])
    q[14] = 1./3. *( p[2] -2* p[6]+ p[5] -2* p[7]+2* p[4]+ p[8]+2* p[3])
    q[8] = p[6] -2* p[5]+ p[1] -2* p[9]+2* p[4]+ p[3]
    q[1] = p[0]
    q[6] = p[0] - p[8]+ p[3]
    q[7] = p[7] -2* p[9]+ p[4] -2* p[8]+2* p[3]+ p[0]
    q[2] = -p[2]+3* p[6] -3* p[5]+ p[1]+3* p[7] -6* p[9]+3* p[4] -3* p[8]+3* p[3]+ p[0]
